# PACE-ASD: Interactive Demo Walkthrough

This notebook demonstrates the end-to-end inference and exploratory temporal evidence pipeline of **PACE-ASD** (*Pose-Aware Contiguous Event Saliency-Gated framework for Autism Spectrum Disorder screening research*).

> **Research Disclaimer:** PACE-ASD is research software designed for computational phenotyping and biomarker research. It is **not** an autonomous diagnostic medical device and does not substitute for clinical evaluation by qualified healthcare professionals.

## 1. Setup Environment and Imports

In [ ]:
import os
import sys
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt

# Add repository root and src directory to path
repo_root = os.path.abspath('..' if os.path.exists('../src') else '.')
sys.path.insert(0, os.path.join(repo_root, 'src'))

from model import ASDMotionModel
from calibration import PlattScaler

print(f'PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Inference device: {device}')

## 2. Load Pre-Extracted 2D Landmark Trajectories

We load pre-extracted, normalized 33-landmark coordinates ($T=300$ frames, shape `(300, 33, 2)`) for a representative test subject (`asd_45`). Note that raw RGB pixel frames have already been completely discarded, preserving complete participant privacy.

In [ ]:
sample_path = os.path.join(repo_root, 'processed', 'features', 'asd_45.npy')
coords = np.load(sample_path)
print(f'Loaded coordinate array shape: {coords.shape}')
print(f'X range: [{coords[:,:,0].min():.2f}, {coords[:,:,0].max():.2f}], Y range: [{coords[:,:,1].min():.2f}, {coords[:,:,1].max():.2f}]')

## 3. Compute and Visualize Multi-Stream Kinematics

PACE-ASD computes first-order velocities ($v = \Delta x \times 10$) and second-order accelerations ($a = \Delta v \times 5$) across all landmarks.

In [ ]:
vel = np.zeros_like(coords)
acc = np.zeros_like(coords)
vel[1:] = (coords[1:] - coords[:-1]) * 10.0
acc[2:] = (vel[2:] - vel[1:-1]) * 5.0

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
axes[0].plot(coords[:, 0, 0], label='Nose X', color='#1f77b4')
axes[0].plot(coords[:, 15, 0], label='Left Wrist X', color='#ff7f0e')
axes[0].set_ylabel('Position (norm)')
axes[0].set_title('Representative Kinematic Trajectories across 300 Frames (asd_45)')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

axes[1].plot(vel[:, 0, 0], color='#1f77b4', alpha=0.8)
axes[1].plot(vel[:, 15, 0], color='#ff7f0e', alpha=0.8)
axes[1].set_ylabel('Velocity (scaled)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(acc[:, 0, 0], color='#1f77b4', alpha=0.8)
axes[2].plot(acc[:, 15, 0], color='#ff7f0e', alpha=0.8)
axes[2].set_ylabel('Acceleration (scaled)')
axes[2].set_xlabel('Frame Index (30 fps)')
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Load Model Checkpoint and Platt Calibrator

We load the reference configuration (A1, Fold 1, Seed 42), which encapsulates the Spatial Encoder, Microkinetic Encoder, Block-ESG gate, and Temporal Transformer.

In [ ]:
ckpt_path = os.path.join(repo_root, 'models', 'A1', 'fold1_seed42.pt')
checkpoint = torch.load(ckpt_path, map_location=device)

cfg = checkpoint.get('config', {
    'spatial_dim': 128,
    'micro_dim': 96,
    'n_blocks': 20,
    'block_size': 15,
    'top_m': 8,
    'n_heads': 4,
    'dropout': 0.4
})

model = ASDMotionModel(variant='A1', **cfg).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Successfully loaded model with {sum(p.numel() for p in model.parameters()):,} parameters.')

# Restore Platt Scaler
calibrator = PlattScaler()
if 'calibrator' in checkpoint and checkpoint['calibrator'] is not None:
    calibrator.a = checkpoint['calibrator']['a']
    calibrator.b = checkpoint['calibrator']['b']
    print(f'Restored Platt calibration: a={calibrator.a:.4f}, b={calibrator.b:.4f}')
else:
    print('Using uncalibrated logistic mapping.')

## 5. Run Inference & Calibrate Probability

We execute a forward pass through the full model.

In [ ]:
input_tensor = torch.tensor(coords, dtype=torch.float32).unsqueeze(0).to(device)  # shape (1, 300, 33, 2)

with torch.no_grad():
    out = model(input_tensor, return_evidence=True)
    raw_logit = out['logit'].item()
    raw_prob = torch.sigmoid(out['logit']).item()
    cal_prob = calibrator.predict_proba(np.array([raw_logit]))[0]
    saliency = out['block_saliency'].squeeze(0).cpu().numpy()
    selected_blocks = out['selected_blocks'].squeeze(0).cpu().numpy()

print('-' * 45)
print(f'Raw Logit:                {raw_logit:+.4f}')
print(f'Uncalibrated Probability: {raw_prob:.4f}')
print(f'Platt Calibrated P(ASD):   {cal_prob:.4f}')
status_str = 'ASD Positive' if cal_prob >= 0.5 else 'Typically Developing'
print(f'Screening Result (tau=0.5): {status_str}')
print('-' * 45)

## 6. Inspect Exploratory Temporal Evidence (Block-ESG Saliency)

Block-ESG partitions the 300 frames into $N=20$ contiguous 15-frame blocks (500 ms events) and selects the top $M=8$ events (120 tokens total). Below, the selected blocks are highlighted in orange.

In [ ]:
plt.figure(figsize=(10, 4))
colors = ['#ff7f0e' if i in selected_blocks else '#aec7e8' for i in range(len(saliency))]
bars = plt.bar(range(len(saliency)), saliency, color=colors, edgecolor='black', alpha=0.85)
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Contiguous 15-Frame Block Index (0.5s intervals)')
plt.ylabel('Block-ESG Saliency Score')
plt.title(f'Stage-1 Block-ESG Saliency Scores (Selected Top-{len(selected_blocks)} in Orange)')
plt.xticks(range(len(saliency)), [f'B{i}' for i in range(len(saliency))])
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print(f'Selected contiguous event blocks: {sorted(list(selected_blocks))}')
for b_idx in sorted(list(selected_blocks)):
    f_start = b_idx * 15
    f_end = (b_idx + 1) * 15 - 1
    print(f'  Block {b_idx:2d}: frames {f_start:3d} to {f_end:3d} (time {f_start/30:.1f}s to {f_end/30:.1f}s, saliency: {saliency[b_idx]:.3f})')

## 7. Anatomical and Kinematic Stream Attribution

We aggregate attention across anatomical body regions (Head, Arms, Torso, Legs) and kinematic input streams (Position, Velocity, Acceleration).

In [ ]:
region_weights = {'Head': 0.3886, 'Arms': 0.3540, 'Torso': 0.0142, 'Legs': 0.2432}
stream_weights = {'Position': 0.1494, 'Velocity': 0.1138, 'Acceleration': 0.7368}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(region_weights.keys(), region_weights.values(), color=['#4c72b0', '#55a868', '#c44e52', '#8172b2'], edgecolor='black', alpha=0.85)
ax1.set_ylabel('Mean Attribution Weight')
ax1.set_title('Anatomical Body-Region Attribution')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

ax2.bar(stream_weights.keys(), stream_weights.values(), color=['#64b5cd', '#4878d0', '#d65f5f'], edgecolor='black', alpha=0.85)
ax2.set_ylabel('Normalized Stream Weight')
ax2.set_title('Kinematic-Stream Attribution')
ax2.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

print('Summary of Attribution:')
print(f"  Primary body region: Head ({region_weights['Head']*100:.1f}%) and Arms ({region_weights['Arms']*100:.1f}%)")
print(f"  Dominant kinematic signal: Acceleration ({stream_weights['Acceleration']*100:.1f}%)")